In [1]:
!pip install faiss-cpu

In [18]:
import subprocess
import time
import threading


# 1. Install required dependency zstd for Ollama installation
!apt-get install zstd -qq

# Install Ollama and the Python library
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama -q
import ollama
# 2. Start Ollama server in the background
def run_ollama():
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

threading.Thread(target=run_ollama, daemon=True).start()
time.sleep(5)  # Give the server a moment to warm up

# 3. Pull a odel (Llama 3.2 is ~2GB)
print("Downloading model...")
ollama.pull('llama3.2')

# 4. Ask a simple question
print("\n--- Model Response ---")
response = ollama.chat(model='llama3.2', messages=[
    {'role': 'user', 'content': 'What is the most interesting fact about space?'}
])

print(response['message']['content'])

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

--- Model Response ---
There are countless fascinating facts about space, and it's hard to pick just one. However, here's a mind-blowing one:

**The Universe is Still Growing!**

In the late 1920s, American astronomer Edwin Hubble made a groundbreaking discovery that revolutionized our understanding of the cosmos. He observed that the light coming from distant galaxies was shifted towards the red end of the spectrum, which meant that those galaxies were moving away from us.

This observation led to the conclusion that the universe is expanding,

In [19]:
!pip install faiss-cpu sentence-transformers -q

In [21]:
import subprocess
import time
import threading
import socket
import faiss
import numpy as np
import ollama
from sentence_transformers import SentenceTransformer


# --- 2. SETUP VECTOR MEMORY (FAISS) ---
# This model converts text into 384-dimensional vectors
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
dimension = 384
index = faiss.IndexFlatL2(dimension)
metadata = [] # Stores the actual text strings

def add_to_memory(text):
    embedding = embed_model.encode([text])
    index.add(np.array(embedding).astype('float32'))
    metadata.append(text)
    print(f"-> [System]: Saved to FAISS memory: '{text}'")

def search_memory(query, threshold=1.1):
    if index.ntotal == 0:
        return None

    query_vector = embed_model.encode([query])
    distances, indices = index.search(np.array(query_vector).astype('float32'), k=1)

    # Only return if the similarity is close enough (lower distance = closer match)
    if distances[0][0] < threshold:
        return metadata[indices[0][0]]
    return None

# --- 3. THE CHAT LOGIC ---
def chat_with_ai(user_input):
    user_input_lower = user_input.lower().strip()

    # Define if the input is a question
    question_words = ['who', 'what', 'where', 'when', 'why', 'how', '?', 'is it', 'can you']
    is_question = any(word in user_input_lower for word in question_words) or user_input_lower.endswith('?')

    # --- STEP A: GREETING ---
    if user_input_lower in ['hi', 'hello', 'hey']:
        return "Hello! I am your personal AI assistant. How can I help you today?"

    # --- STEP B: LEARNING (If statement contains 'is/are' and is NOT a question) ---
    if (" is " in user_input_lower or " are " in user_input_lower) and not is_question:
        add_to_memory(user_input)
        return "Got it! I've updated my memory with that information."

    # --- STEP C: RETRIEVAL ---
    context = search_memory(user_input)

    if context:
        print("(Using Memory Retrieval...)")
        full_prompt = f"Using this personal context: '{context}'. Answer the user: {user_input}"
    else:
        print("(Querying Llama 3.2 general knowledge...)")
        full_prompt = user_input

    # --- STEP D: GENERATION ---
    response = ollama.chat(model='llama3.2', messages=[{'role': 'user', 'content': full_prompt}])
    return response['message']['content']

# --- 4. EXECUTION LOOP ---
print("\n--- Mini Gemini/ChatGPT Ready ---")
print("(Type 'exit' to stop)")

while True:
    user_text = input("You: ")
    if user_text.lower() in ['exit', 'quit']:
        break

    response = chat_with_ai(user_text)
    print(f"AI: {response}\n")

Initializing Ollama server...
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Ollama is running. Pulling Llama 3.2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Mini Gemini/ChatGPT Ready ---
(Type 'exit' to stop)
You: Hi
AI: Hello! I am your personal AI assistant. How can I help you today?

You: Who is Alice ?
(Querying Llama 3.2 general knowledge...)
AI: Alice can refer to several people and characters, so I'll provide a few possibilities:

1. **Lewis Carroll's "Alice"**: In Lewis Carroll's classic novel "Alice's Adventures in Wonderland", Alice is the protagonist, a young girl who falls down a rabbit hole and enters a fantastical world called Wonderland.
2. **The Beatles' "Alice"**: The Beatles wrote and recorded a song called "Lucy in the Sky with Diamonds", which features lyrics that mention an "alice". However, I couldn't find any specific information on a character named Alice being associated with this song.
3. **Royal families: Princess Alice**: There have been several Royal Highnesses named Alice throughout history, including:
 * Princess Alice of Albany (1843-1878), wife of the Earl of Sefton
 * Princess Alice, Duchess of Saxe-M